# Controller Trace vs Ground-Truth Validation Trace Analysis

This notebook compares validation runs produced by `run_scripts/run_controller_validation_eval.sh` against mined validation oracle traces from `outputs/oracle_dataset/validation`. The goal is not only to ask whether the controller answered correctly, but whether it reproduced the oracle reasoning policy: action type sequence, visual index choices, trace length, stop behavior, and stepwise divergence.

The most useful read is usually a three-way comparison: answer correctness, trace similarity, and where the first mismatch occurs.

In [ ]:
from collections import Counter, defaultdict
from itertools import zip_longest
from pathlib import Path
import json
import math
import re

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd
from IPython.display import display

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "analysis":
    PROJECT_ROOT = PROJECT_ROOT.parent

RUN_ROOT = PROJECT_ROOT / "outputs" / "controller_sft_m3cot_test_sweeps"
EVAL_ROOT = RUN_ROOT / "eval_validation"
TRACE_ROOT = PROJECT_ROOT / "outputs" / "oracle_dataset" / "validation"
MANIFEST_PATH = RUN_ROOT / "controller_runs.jsonl"

# Leave as None to infer from trace_source names such as global_test or coarse_test_replayed_global.
ORACLE_CONTEXT_OVERRIDE = None  # e.g. "global" or "coarse"
ORACLE_MINED_BY_PREFERENCE = "lvar"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("EVAL_ROOT:", EVAL_ROOT, "exists=", EVAL_ROOT.exists())
print("TRACE_ROOT:", TRACE_ROOT, "exists=", TRACE_ROOT.exists())
print("MANIFEST_PATH:", MANIFEST_PATH, "exists=", MANIFEST_PATH.exists())


## 1. Load Controller Runs And Oracle Traces

Controller validation predictions are expected at `eval_validation/{trace_source}/{variant}/m3cot_validation_predictions.jsonl`. Ground-truth traces are expected as `m3cot_validation_traces_{mined_by}_{context}.jsonl`. If there are several oracle files for the same context, the notebook prefers `mined_by=lvar`, then falls back to the first available file.

In [ ]:
def read_json(path: Path):
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def read_jsonl(path: Path):
    rows = []
    bad = []
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            stripped = line.strip()
            if not stripped:
                continue
            try:
                rows.append(json.loads(stripped))
            except json.JSONDecodeError as exc:
                bad.append({"path": str(path), "line_number": line_number, "error": str(exc)})
    return rows, bad


def prediction_summary_path(prediction_path: Path) -> Path:
    return prediction_path.with_name(f"{prediction_path.stem}_summary.json")


def parse_oracle_trace_path(path: Path):
    match = re.search(r"m3cot_validation_traces_(?P<mined_by>.+?)_(?P<context>global|coarse)\.jsonl$", path.name)
    if not match:
        return None, None
    return match.group("mined_by"), match.group("context")


def infer_oracle_context(trace_source: str):
    if ORACLE_CONTEXT_OVERRIDE:
        return ORACLE_CONTEXT_OVERRIDE
    name = str(trace_source).lower()
    if "coarse" in name:
        return "coarse"
    if "global" in name:
        return "global"
    return None


def discover_controller_runs(root: Path) -> pd.DataFrame:
    rows = []
    for prediction_path in sorted(root.glob("*/*/m3cot_validation_predictions.jsonl")):
        rel = prediction_path.relative_to(root).parts
        trace_source, variant = rel[0], rel[1]
        summary_path = prediction_summary_path(prediction_path)
        summary = read_json(summary_path) if summary_path.exists() else {}
        metrics = summary.get("metrics", {}) or {}
        rows.append({
            "trace_source": trace_source,
            "variant": variant,
            "oracle_context": infer_oracle_context(trace_source),
            "prediction_path": str(prediction_path),
            "summary_path": str(summary_path) if summary_path.exists() else None,
            "summary_exists": summary_path.exists(),
            "total": metrics.get("total"),
            "correct": metrics.get("correct"),
            "accuracy": metrics.get("accuracy"),
            "avg_controller_actions": metrics.get("avg_controller_actions"),
            "avg_controller_tokens": metrics.get("avg_controller_tokens"),
            "avg_output_tokens": metrics.get("avg_output_tokens"),
            "controller_checkpoint": summary.get("controller_checkpoint"),
        })
    return pd.DataFrame(rows)


def discover_oracle_traces(root: Path):
    manifest_rows = []
    by_context = defaultdict(list)
    for path in sorted(root.rglob("m3cot_validation_traces_*_*.jsonl")):
        mined_by, context = parse_oracle_trace_path(path)
        if context is None:
            continue
        rows, bad = read_jsonl(path)
        summary_path = path.with_name(path.stem + "_summary.json")
        summary = read_json(summary_path) if summary_path.exists() else {}
        record = {
            "path": path,
            "mined_by": mined_by,
            "context": context,
            "rows": rows,
            "bad_rows": bad,
            "summary": summary,
        }
        by_context[context].append(record)
        manifest_rows.append({
            "mined_by": mined_by,
            "context": context,
            "num_rows": len(rows),
            "bad_rows": len(bad),
            "summary_exists": summary_path.exists(),
            "path": str(path),
        })
    return pd.DataFrame(manifest_rows), by_context


controller_runs = discover_controller_runs(EVAL_ROOT) if EVAL_ROOT.exists() else pd.DataFrame()
oracle_manifest, oracle_by_context = discover_oracle_traces(TRACE_ROOT) if TRACE_ROOT.exists() else (pd.DataFrame(), {})

print(f"Controller runs discovered: {len(controller_runs)}")
print(f"Oracle trace files discovered: {len(oracle_manifest)}")
display(controller_runs.sort_values(["trace_source", "variant"]) if not controller_runs.empty else controller_runs)
display(oracle_manifest.sort_values(["context", "mined_by"]) if not oracle_manifest.empty else oracle_manifest)


In [ ]:
def choose_oracle_record(context: str):
    candidates = oracle_by_context.get(context, [])
    if not candidates:
        return None
    preferred = [r for r in candidates if str(r["mined_by"]) == ORACLE_MINED_BY_PREFERENCE]
    return (preferred or candidates)[0]


oracle_maps = {}
for context in sorted(oracle_by_context):
    record = choose_oracle_record(context)
    if record is None:
        continue
    oracle_maps[context] = {str(row.get("example_id", row.get("id"))): row for row in record["rows"]}
    print(f"Using oracle for context={context}: mined_by={record['mined_by']} path={record['path']}")


## 2. Trace Normalization And Metrics

The controller prediction trace is primitive-step oriented (`action`, optional `patch_index`/`region_index`). The oracle trace is usually stored as primitive actions under `trace`; if that is absent, the notebook flattens non-empty `decisions`. Metrics are computed both with visual indices (`PATCH:5`) and type-only labels (`PATCH`).

Key metrics:

- `exact_signature_match`: full primitive sequence matches including visual indices.
- `exact_type_match`: full primitive action-type sequence matches while ignoring indices.
- `type_edit_similarity`: normalized Levenshtein similarity for action types; good for near misses.
- `prefix_type_recall` / `prefix_type_precision`: how much of the shared beginning matches before the first divergence.
- `visual_index_accuracy_aligned`: index accuracy only at aligned PATCH/REGION positions where both traces chose the same visual action type.
- `patch_jaccard` / `region_jaccard`: set overlap of visual evidence visited, ignoring order.
- `controller_correct`: final decoded answer correctness from the controller run.

In [ ]:
PAD = "<PAD>"


def _single_index(action, index_name_a, index_name_b=None):
    if action.get(index_name_a) is not None:
        return int(action[index_name_a])
    if index_name_b and action.get(index_name_b) is not None:
        return int(action[index_name_b])
    return None


def normalize_action(action, source: str):
    if source == "controller":
        action_type = str(action.get("action") or action.get("type") or "").upper()
        patch_index = _single_index(action, "patch_index", "patch_idx")
        region_index = _single_index(action, "region_index", "region_idx")
        patch_indices = action.get("patch_indices") or []
        region_indices = action.get("region_indices") or []
    else:
        action_type = str(action.get("type") or action.get("action") or "").upper()
        patch_index = _single_index(action, "patch_idx", "patch_index")
        region_index = _single_index(action, "region_idx", "region_index")
        patch_indices = action.get("patch_indices") or []
        region_indices = action.get("region_indices") or []

    if action_type == "PATCH":
        if patch_index is not None:
            signature = f"PATCH:{patch_index}"
            visual_index = patch_index
        elif patch_indices:
            vals = tuple(int(v) for v in patch_indices)
            signature = "PATCH:" + ",".join(map(str, vals))
            visual_index = vals
        else:
            signature = "PATCH"
            visual_index = None
    elif action_type == "REGION":
        if region_index is not None:
            signature = f"REGION:{region_index}"
            visual_index = region_index
        elif region_indices:
            vals = tuple(int(v) for v in region_indices)
            signature = "REGION:" + ",".join(map(str, vals))
            visual_index = vals
        else:
            signature = "REGION"
            visual_index = None
    else:
        signature = action_type or "UNKNOWN"
        visual_index = None

    return {"type": action_type or "UNKNOWN", "signature": signature, "visual_index": visual_index}


def oracle_actions_from_row(row):
    raw_trace = row.get("trace") or []
    if raw_trace:
        return [dict(action) for action in raw_trace]
    actions = []
    for decision in row.get("decisions") or []:
        actions.extend(dict(action) for action in (decision.get("actions") or []))
    return actions


def normalize_trace(row, source: str):
    actions = row.get("trace") or [] if source == "controller" else oracle_actions_from_row(row)
    return [normalize_action(action, source=source) for action in actions]


def seq_types(trace):
    return [a["type"] for a in trace]


def seq_signatures(trace):
    return [a["signature"] for a in trace]


def levenshtein(a, b):
    previous = list(range(len(b) + 1))
    for i, av in enumerate(a, start=1):
        current = [i]
        for j, bv in enumerate(b, start=1):
            current.append(min(
                previous[j] + 1,
                current[j - 1] + 1,
                previous[j - 1] + (av != bv),
            ))
        previous = current
    return previous[-1]


def longest_common_prefix(a, b):
    count = 0
    for av, bv in zip(a, b):
        if av != bv:
            break
        count += 1
    return count


def counter_f1(a, b):
    ca, cb = Counter(a), Counter(b)
    overlap = sum((ca & cb).values())
    precision = overlap / sum(ca.values()) if ca else np.nan
    recall = overlap / sum(cb.values()) if cb else np.nan
    if not precision or not recall or math.isnan(precision) or math.isnan(recall):
        f1 = np.nan if math.isnan(precision) or math.isnan(recall) else 0.0
    else:
        f1 = 2 * precision * recall / (precision + recall)
    return precision, recall, f1


def visual_set(trace, action_type):
    values = set()
    for action in trace:
        if action["type"] != action_type:
            continue
        value = action["visual_index"]
        if isinstance(value, tuple):
            values.update(value)
        elif value is not None:
            values.add(value)
    return values


def jaccard(a, b):
    if not a and not b:
        return np.nan
    return len(a & b) / len(a | b) if (a or b) else np.nan


def compare_trace_pair(pred_row, oracle_row, run_meta):
    pred_trace = normalize_trace(pred_row, source="controller")
    oracle_trace = normalize_trace(oracle_row, source="oracle")
    pred_types, oracle_types = seq_types(pred_trace), seq_types(oracle_trace)
    pred_sigs, oracle_sigs = seq_signatures(pred_trace), seq_signatures(oracle_trace)
    max_type_len = max(len(pred_types), len(oracle_types), 1)
    max_sig_len = max(len(pred_sigs), len(oracle_sigs), 1)
    lcp_type = longest_common_prefix(pred_types, oracle_types)
    lcp_sig = longest_common_prefix(pred_sigs, oracle_sigs)
    type_precision, type_recall, type_f1 = counter_f1(pred_types, oracle_types)

    aligned_visual_total = 0
    aligned_visual_correct = 0
    first_mismatch_step = None
    for idx, (p, o) in enumerate(zip_longest(pred_trace, oracle_trace), start=1):
        p_sig = p["signature"] if p is not None else PAD
        o_sig = o["signature"] if o is not None else PAD
        if first_mismatch_step is None and p_sig != o_sig:
            first_mismatch_step = idx
        if p is not None and o is not None and p["type"] == o["type"] and p["type"] in {"PATCH", "REGION"}:
            if not isinstance(p["visual_index"], tuple) and not isinstance(o["visual_index"], tuple):
                if p["visual_index"] is not None and o["visual_index"] is not None:
                    aligned_visual_total += 1
                    aligned_visual_correct += int(p["visual_index"] == o["visual_index"])

    pred_patches, oracle_patches = visual_set(pred_trace, "PATCH"), visual_set(oracle_trace, "PATCH")
    pred_regions, oracle_regions = visual_set(pred_trace, "REGION"), visual_set(oracle_trace, "REGION")

    return {
        **run_meta,
        "example_id": str(pred_row.get("example_id")),
        "controller_correct": bool(pred_row.get("correct")),
        "gold_answer": pred_row.get("gold_answer"),
        "decoded_answer": pred_row.get("generated_text"),
        "pred_len": len(pred_types),
        "oracle_len": len(oracle_types),
        "length_delta": len(pred_types) - len(oracle_types),
        "pred_stop": bool(pred_types and pred_types[-1] == "STOP"),
        "oracle_stop": bool(oracle_types and oracle_types[-1] == "STOP"),
        "exact_signature_match": pred_sigs == oracle_sigs,
        "exact_type_match": pred_types == oracle_types,
        "type_edit_distance": levenshtein(pred_types, oracle_types),
        "signature_edit_distance": levenshtein(pred_sigs, oracle_sigs),
        "type_edit_similarity": 1.0 - levenshtein(pred_types, oracle_types) / max_type_len,
        "signature_edit_similarity": 1.0 - levenshtein(pred_sigs, oracle_sigs) / max_sig_len,
        "lcp_type": lcp_type,
        "lcp_signature": lcp_sig,
        "prefix_type_recall": lcp_type / len(oracle_types) if oracle_types else np.nan,
        "prefix_type_precision": lcp_type / len(pred_types) if pred_types else np.nan,
        "first_mismatch_step": first_mismatch_step,
        "type_multiset_precision": type_precision,
        "type_multiset_recall": type_recall,
        "type_multiset_f1": type_f1,
        "visual_index_accuracy_aligned": aligned_visual_correct / aligned_visual_total if aligned_visual_total else np.nan,
        "aligned_visual_positions": aligned_visual_total,
        "patch_jaccard": jaccard(pred_patches, oracle_patches),
        "region_jaccard": jaccard(pred_regions, oracle_regions),
        "pred_types": pred_types,
        "oracle_types": oracle_types,
        "pred_signatures": pred_sigs,
        "oracle_signatures": oracle_sigs,
    }


In [ ]:
comparison_rows = []
missing_oracle_rows = []

for run in controller_runs.to_dict("records") if not controller_runs.empty else []:
    context = run.get("oracle_context")
    oracle_map = oracle_maps.get(context, {}) if context else {}
    predictions, bad_predictions = read_jsonl(Path(run["prediction_path"]))
    if bad_predictions:
        print(f"Warning: {len(bad_predictions)} bad prediction rows in {run['prediction_path']}")
    run_meta = {
        "trace_source": run["trace_source"],
        "variant": run["variant"],
        "oracle_context": context,
        "prediction_path": run["prediction_path"],
    }
    for pred_row in predictions:
        example_id = str(pred_row.get("example_id"))
        oracle_row = oracle_map.get(example_id)
        if oracle_row is None:
            missing_oracle_rows.append({**run_meta, "example_id": example_id})
            continue
        comparison_rows.append(compare_trace_pair(pred_row, oracle_row, run_meta))

comparison_df = pd.DataFrame(comparison_rows)
missing_oracle_df = pd.DataFrame(missing_oracle_rows)
print(f"Compared examples: {len(comparison_df):,}")
print(f"Missing oracle rows: {len(missing_oracle_df):,}")
if not missing_oracle_df.empty:
    display(missing_oracle_df.head(20))

display(comparison_df.head() if not comparison_df.empty else comparison_df)


## 3. Run-Level Summary

Start here. A strong controller should combine high `controller_accuracy`, high edit similarity, high prefix agreement, low length delta, and good visual index overlap. If answer accuracy is high but trace agreement is low, the controller may be finding alternate reasoning paths or exploiting answer priors. If trace agreement is high but answer accuracy is low, the oracle trace itself may not be sufficient under the controller/VLM checkpoint pairing or the answer decoder may be the bottleneck.

In [ ]:
if comparison_df.empty:
    print("No comparisons to summarize yet. Run controller validation and validation oracle mining/eval first.")
else:
    summary_df = (
        comparison_df
        .groupby(["trace_source", "variant", "oracle_context"], observed=True)
        .agg(
            examples=("example_id", "nunique"),
            controller_accuracy=("controller_correct", "mean"),
            exact_signature_match=("exact_signature_match", "mean"),
            exact_type_match=("exact_type_match", "mean"),
            type_edit_similarity=("type_edit_similarity", "mean"),
            signature_edit_similarity=("signature_edit_similarity", "mean"),
            prefix_type_recall=("prefix_type_recall", "mean"),
            prefix_type_precision=("prefix_type_precision", "mean"),
            type_multiset_f1=("type_multiset_f1", "mean"),
            visual_index_accuracy_aligned=("visual_index_accuracy_aligned", "mean"),
            patch_jaccard=("patch_jaccard", "mean"),
            region_jaccard=("region_jaccard", "mean"),
            pred_len_mean=("pred_len", "mean"),
            oracle_len_mean=("oracle_len", "mean"),
            length_delta_mean=("length_delta", "mean"),
            pred_stop_rate=("pred_stop", "mean"),
        )
        .reset_index()
        .sort_values(["trace_source", "controller_accuracy", "type_edit_similarity"], ascending=[True, False, False])
    )
    display(summary_df.style.format({
        "controller_accuracy": "{:.2%}",
        "exact_signature_match": "{:.2%}",
        "exact_type_match": "{:.2%}",
        "type_edit_similarity": "{:.3f}",
        "signature_edit_similarity": "{:.3f}",
        "prefix_type_recall": "{:.3f}",
        "prefix_type_precision": "{:.3f}",
        "type_multiset_f1": "{:.3f}",
        "visual_index_accuracy_aligned": "{:.2%}",
        "patch_jaccard": "{:.3f}",
        "region_jaccard": "{:.3f}",
        "pred_len_mean": "{:.2f}",
        "oracle_len_mean": "{:.2f}",
        "length_delta_mean": "{:+.2f}",
        "pred_stop_rate": "{:.2%}",
    }, na_rep="-").hide(axis="index"))


In [ ]:
if not comparison_df.empty:
    plot_metrics = ["controller_accuracy", "type_edit_similarity", "prefix_type_recall", "patch_jaccard", "region_jaccard"]
    melted = summary_df.melt(
        id_vars=["trace_source", "variant", "oracle_context"],
        value_vars=plot_metrics,
        var_name="metric",
        value_name="value",
    )
    melted["run"] = melted["trace_source"].astype(str) + " / " + melted["variant"].astype(str)
    if HAS_SEABORN:
        g = sns.catplot(data=melted, x="run", y="value", hue="metric", kind="bar", height=4.5, aspect=1.8)
        g.set_axis_labels("Controller run", "Score")
        for ax in g.axes.flat:
            ax.tick_params(axis="x", rotation=35)
            ax.yaxis.set_major_formatter(PercentFormatter(1.0))
        plt.tight_layout()
        plt.show()
    else:
        ax = melted.pivot_table(index="run", columns="metric", values="value").plot(kind="bar", figsize=(12, 5))
        ax.set_ylabel("Score")
        ax.yaxis.set_major_formatter(PercentFormatter(1.0))
        plt.xticks(rotation=35, ha="right")
        plt.tight_layout()
        plt.show()


## 4. Stepwise Agreement And Confusion

These plots answer: at which action step does the controller diverge? Because traces have variable length, each step includes a coverage count. Late steps can have fewer examples and should be interpreted with that in mind.

In [ ]:
step_rows = []
if not comparison_df.empty:
    for row in comparison_df.to_dict("records"):
        max_len = max(len(row["pred_types"]), len(row["oracle_types"]))
        for step in range(max_len):
            pred_type = row["pred_types"][step] if step < len(row["pred_types"]) else PAD
            oracle_type = row["oracle_types"][step] if step < len(row["oracle_types"]) else PAD
            pred_sig = row["pred_signatures"][step] if step < len(row["pred_signatures"]) else PAD
            oracle_sig = row["oracle_signatures"][step] if step < len(row["oracle_signatures"]) else PAD
            step_rows.append({
                "trace_source": row["trace_source"],
                "variant": row["variant"],
                "oracle_context": row["oracle_context"],
                "example_id": row["example_id"],
                "step": step + 1,
                "pred_type": pred_type,
                "oracle_type": oracle_type,
                "type_match": pred_type == oracle_type,
                "signature_match": pred_sig == oracle_sig,
                "controller_correct": row["controller_correct"],
            })

step_df = pd.DataFrame(step_rows)
if step_df.empty:
    print("No step rows available.")
else:
    step_summary = (
        step_df.groupby(["trace_source", "variant", "oracle_context", "step"], observed=True)
        .agg(
            examples=("example_id", "nunique"),
            type_match_rate=("type_match", "mean"),
            signature_match_rate=("signature_match", "mean"),
        )
        .reset_index()
    )
    display(step_summary.head(40).style.format({
        "type_match_rate": "{:.2%}",
        "signature_match_rate": "{:.2%}",
    }).hide(axis="index"))


In [ ]:
if not step_df.empty:
    for (trace_source, variant), sub in step_summary.groupby(["trace_source", "variant"], observed=True):
        fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
        axes[0].plot(sub["step"], sub["examples"], marker="o")
        axes[0].set(title=f"Coverage by step: {trace_source}/{variant}", xlabel="Trace step", ylabel="Examples")
        axes[0].xaxis.set_major_locator(plt.MaxNLocator(integer=True))
        axes[1].plot(sub["step"], sub["type_match_rate"], marker="o", label="type")
        axes[1].plot(sub["step"], sub["signature_match_rate"], marker="o", label="type + index")
        axes[1].set(title="Stepwise agreement", xlabel="Trace step", ylabel="Match rate")
        axes[1].yaxis.set_major_formatter(PercentFormatter(1.0))
        axes[1].xaxis.set_major_locator(plt.MaxNLocator(integer=True))
        axes[1].legend()
        plt.tight_layout()
        plt.show()


In [ ]:
if not step_df.empty:
    for (trace_source, variant), sub in step_df.groupby(["trace_source", "variant"], observed=True):
        confusion = pd.crosstab(sub["oracle_type"], sub["pred_type"], normalize="index")
        display(confusion.style.format("{:.2%}").set_caption(f"Oracle -> controller action confusion: {trace_source}/{variant}"))
        if HAS_SEABORN:
            plt.figure(figsize=(8, 5))
            sns.heatmap(confusion, annot=True, fmt=".0%", cmap="Blues", vmin=0, vmax=1)
            plt.title(f"Action confusion: {trace_source}/{variant}")
            plt.xlabel("Controller action")
            plt.ylabel("Oracle action")
            plt.tight_layout()
            plt.show()


## 5. Correctness vs Trace Agreement

This section shows whether trace fidelity predicts final answer correctness. Interesting cases are examples that are correct despite low trace similarity and examples that are wrong despite high trace similarity.

In [ ]:
if not comparison_df.empty:
    agreement_cols = [
        "type_edit_similarity",
        "signature_edit_similarity",
        "prefix_type_recall",
        "type_multiset_f1",
        "patch_jaccard",
        "region_jaccard",
    ]
    by_correct = (
        comparison_df
        .groupby(["trace_source", "variant", "controller_correct"], observed=True)[agreement_cols]
        .mean()
        .reset_index()
    )
    display(by_correct.style.format({col: "{:.3f}" for col in agreement_cols}).hide(axis="index"))

    if HAS_SEABORN:
        g = sns.displot(
            data=comparison_df,
            x="type_edit_similarity",
            hue="controller_correct",
            col="variant",
            row="trace_source",
            kind="hist",
            bins=20,
            height=3,
            aspect=1.25,
            stat="density",
            common_norm=False,
        )
        g.set_axis_labels("Type edit similarity", "Density")
        plt.show()


In [ ]:
if not comparison_df.empty:
    interesting = comparison_df.copy()
    interesting["case"] = np.select(
        [
            interesting["controller_correct"] & (interesting["type_edit_similarity"] < 0.5),
            (~interesting["controller_correct"]) & (interesting["type_edit_similarity"] > 0.8),
            interesting["exact_type_match"] & (~interesting["exact_signature_match"]),
        ],
        [
            "correct_low_trace_similarity",
            "wrong_high_trace_similarity",
            "right_types_wrong_indices",
        ],
        default="",
    )
    interesting = interesting[interesting["case"] != ""]
    cols = [
        "case", "trace_source", "variant", "example_id", "controller_correct", "gold_answer",
        "type_edit_similarity", "signature_edit_similarity", "first_mismatch_step",
        "pred_types", "oracle_types", "pred_signatures", "oracle_signatures",
    ]
    display(interesting.sort_values(["case", "type_edit_similarity"], ascending=[True, False])[cols].head(50))


## 6. How To Read The Results Critically

Prefer `type_edit_similarity` and `prefix_type_recall` over exact match alone; exact match is intentionally harsh when visual indices differ by one step or the controller chooses a shorter equivalent trace. Use `patch_jaccard` and `region_jaccard` to test whether the controller visits the same evidence even if order differs. Stepwise coverage and confusion reveal systematic failure modes: early `STOP`, overuse of `THINK`, skipping visual actions, or selecting `PATCH` where the oracle selected `REGION`.

The highest-value error slices are: correct answers with low trace agreement, wrong answers with high trace agreement, and exact type matches with wrong visual indices. Those distinguish alternate successful policies, answer-decoder limitations, and visual grounding failures.